In [2]:
print("Tool Calling")

Tool Calling


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")

In [4]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

d:\Playground\Agents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# define tools
from llama_index.core.tools import FunctionTool

def add(x: int, y:int) -> int:
    """
    Args: x, y integers
    Returns: Addition Int
    """
    return x+y

def mystery(x:int, y:int) -> int:
    """
    Args: x, y integers
    Returns: value of operation in int    
    """
    return (x-y)/(x+y)

add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [9]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash'
)

response = llm.predict_and_call(
    [add_tool, mystery_tool],
    "Tell me output of mystery function on 2 and 9", verbose=True
)

print(response)

=== Calling Function ===
Calling function: mystery with args: {"y": 9.0, "x": 2.0}
=== Function Output ===
-0.6363636363636364
-0.6363636363636364


In [ ]:
# Load Data
from llama_index.core import SimpleDirectoryReader
document = SimpleDirectoryReader(input_files=['../Data/CAG.pdf']).load_data()

In [11]:
# Split data
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes  =splitter.get_nodes_from_documents(document)

In [13]:
print(nodes[0].get_content(metadata_mode='all'))

page_label: 1
file_name: CAG.pdf
file_path: ..\Data\CAG.pdf
file_type: application/pdf
file_size: 116336
creation_date: 2025-03-13
last_modified_date: 2025-01-19

arXiv:2412.15605v1  [cs.CL]  20 Dec 2024
Don’t Do RAG:
When Cache-Augmented Generation is All You Need for
Knowledge Tasks
Brian J Chan ∗
Chao-Ting Chen∗
Jui-Hung Cheng ∗
Department of Computer Science
National Chengchi University
Taipei, Taiwan
{110703065,110703038,110703007}@nccu.edu.tw
Hen-Hsen Huang
Insititue of Information Science
Academia Sinica
Taipei, Taiwan
hhhuang@iis.sinica.edu.tw
Abstract
Retrieval-augmented generation (RAG) has gained tractionas a
powerful approach for enhancing language models by integra ting
external knowledge sources. However, RAG introduces chall enges
such as retrieval latency, potential errors in document sel ection,
and increased system complexity. With the advent of large la n-
guage models (LLMs) featuring signiﬁcantly extended conte xt win-
dows, this paper proposes an alternative parad

In [12]:
# embedding model
from llama_index.embeddings.gemini import GeminiEmbedding

embed_model = GeminiEmbedding(model='model/embedding-001')

In [18]:
# Define vector store index
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes, embed_model=embed_model)
query_engine = vector_index.as_query_engine(similarity_top_k=2, llm=llm)

In [21]:
# Metadata filtering
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    llm=llm,
    filters=MetadataFilters.from_dicts(
        [
            {'key':'page_label', 'value': '2'}
        ]
    )
)


response = query_engine.query(
    "explain methodology in CAG"
)

print(response)

The CAG framework uses the extended context capabilities of long-context LLMs to enable retrieval-free knowledge integration. The framework operates in three phases:

1.  **External Knowledge Preloading**: A collection of documents is preprocessed and formatted to fit within the model’s extended context window. The LLM then processes the documents, transforming them into a precomputed KV cache, which is stored for future use. The computational cost of processing is incurred only once, regardless of the number of subsequent queries.
2.  **Inference**: During inference, the precomputed KV cache is loaded alongside the user’s query. The LLM uses this cached context to generate responses. By preloading the external knowledge, this phase eliminates retrieval latency and reduces risks of errors or omissions that arise from dynamic retrieval.
3.  **Cache Reset**: To maintain system performance across multiple inference sessions, the KV cache, stored in memory, can be reset efficiently by trun

In [22]:
for n in response.source_nodes:
    print(n.metadata)

{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}
{'page_label': '2', 'file_name': 'CAG.pdf', 'file_path': '..\\Data\\CAG.pdf', 'file_type': 'application/pdf', 'file_size': 116336, 'creation_date': '2025-03-13', 'last_modified_date': '2025-01-19'}


In [36]:
from llama_index.llms.gemini import Gemini

llm = Gemini(
    model = 'models/gemini-2.0-flash',
    temperature=0
)

In [46]:
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [48]:
response = llm.predict_and_call(
    [vector_query], 
    "What is the experimental setup in CAG as described on page 3?", 
    verbose=True
)

print(response)

AttributeError: 'function' object has no attribute 'metadata'